# Interactive Train + Analysis Notebook

Edit the first code cell only. Then run all cells. This notebook runs the same flow as `main.py` (train + analysis) without shell subprocess calls.


In [ ]:
# User-editable block
NOTEBOOK_CONFIG = {
    "flags": {
        "path": "sample_transformer_addition",
        "train": True,
        "analysis": True,
        "verbose": False,
        "device": "cpu"  # e.g. 'cpu', 'cuda', 'cuda:0'
    },
    "experiment": {
        "operations": ["addition"],
        "model_type": "transformer",
        "data": {
            "modulo": 31,
            "nterms": 2,
            "train_frac": 0.3,
            "data_seed": 598
        },
        "analysis": {
            "fourier_norm_threshold": 0.1,
            "freq_threshold_top_fraction": 0.1,
            "freq_threshold_min_components": 4,
            "freq_threshold_fixed": 0.1
        },
        "model": {
            "model_type": "transformer",
            "embedding_dim": 256,
            "n_heads": 1,
            "n_layers": 1,
            "dim_feedforward": 256,
            "dropout": 0.1,
            "max_len": 128,
            "fixed_positional_encoding": False,
            "activation": "relu",
            "norm_first": False,
            "mask": True,
            "tie_weights": False
        },
        "training": {
            "epochs": 10,
            "lr": 1.0e-3,
            "wd": 5.0e-5,
            "beta1": 0.9,
            "beta2": 0.99,
            "use_scheduler": True,
            "lr_decay_interval": 5000,
            "lr_decay": 0.1,
            "save_every": 100,
            "batch_mode": "full",
            "batch_size": 512,
            "training_target": "last_token",
            "cot_prefix_weight": 1.0,
            "cot_final_weight": 1.0,
            "fourier_reg_mode": None,
            "fourier_reg_coefficient": 0.0,
            "fourier_reg_groups": 2,
            "dataloader_num_workers": 0
        }
    }
}


In [ ]:
from pathlib import Path
import copy
import os
import yaml

flags = NOTEBOOK_CONFIG["flags"]
experiment = copy.deepcopy(NOTEBOOK_CONFIG["experiment"])

# Resolve repo root so notebook works whether launched from repo root or samples/
cwd = Path.cwd().resolve()
REPO_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "main.py").exists() and (candidate / "config").is_dir():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repository root (expected main.py and config/).")

os.chdir(REPO_ROOT)
print(f"Using repo root: {REPO_ROOT}")

run_dir = (REPO_ROOT / flags["path"]).resolve()
run_dir.mkdir(parents=True, exist_ok=True)

experiment.setdefault("training", {})["save_dir"] = str(run_dir)
config_path = run_dir / "config.yaml"
with open(config_path, "w") as f:
    yaml.safe_dump(experiment, f, sort_keys=False)

print(f"Wrote config: {config_path}")
print(f"Flags: {flags}")


In [ ]:
import os
import sys
from pathlib import Path

# Re-bind setup variables for static analysis and out-of-order execution safety
required_names = ("REPO_ROOT", "run_dir", "experiment", "flags")
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        f"Missing notebook setup variables: {missing_names}. Run the setup cell first."
    )

REPO_ROOT = Path(globals()["REPO_ROOT"])
run_dir = Path(globals()["run_dir"])
experiment = globals()["experiment"]
flags = globals()["flags"]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config import Config
from training import train
from utils import (
    get_device,
    create_dataset,
    create_model,
    print_dataset_info,
    evaluate_model,
    load_model,
    plot_training_curves,
    resolve_checkpoint,
    extract_checkpoint_accuracies,
)
from analysis.rnn import ablation as rnn_ablation
from analysis.rnn import fourier as rnn_fourier
from analysis.rnn import svd as rnn_svd
from analysis.rnn import trig as rnn_trig
from analysis.transformer import ablation as transformer_ablation
from analysis.transformer import fourier as transformer_fourier
from analysis.transformer import svd as transformer_svd
from analysis.transformer import trig as transformer_trig

ops = sorted(experiment.get("operations", ["addition"]))
model_type = experiment.get("model_type", "rnn")

config, resolved_path = Config.resolve(
    str(run_dir),
    ops,
    model_type,
    analysis_only=flags["analysis"] and not flags["train"],
)

require_cuda = bool(flags.get("device") and str(flags.get("device")).startswith("cuda"))
device = get_device(flags.get("device"), require_cuda=require_cuda)
print(f"Using device: {device}")

dataset = create_dataset(config.operations, config.data.to_dict(), device)
model = create_model(config.model.to_dict(), device=device)

ops_str = ", ".join(op.capitalize() for op in config.operations)
print(f"\nCreated Modular {ops_str} {config.model_type.upper()} Model and Dataset:\n")
print_dataset_info(dataset)

if flags["verbose"]:
    print("\nModel:")
    print(model)
    print("\nConfiguration:")
    print(config)

if flags["train"]:
    final_checkpoint = train(model, dataset, config.training)
    plot_training_curves(final_checkpoint, resolved_path)

if flags["analysis"]:
    print("\n=== Fourier Spectrum Analysis Report ===\n")

    ckpt_path = resolve_checkpoint(resolved_path)
    model, checkpoint = load_model(ckpt_path, config.model.to_dict(), device=str(device))
    dataset = dataset.to_device(str(device))
    analysis_cache = {}

    if config.model_type == "transformer":
        fourier_analysis = transformer_fourier
        svd_analysis = transformer_svd
        ablation_analysis = transformer_ablation
        trig_analysis = transformer_trig
    else:
        fourier_analysis = rnn_fourier
        svd_analysis = rnn_svd
        ablation_analysis = rnn_ablation
        trig_analysis = rnn_trig

    train_acc, test_acc, epoch = extract_checkpoint_accuracies(checkpoint)

    os.makedirs(os.path.join(resolved_path, "figures"), exist_ok=True)

    print(f"Epoch: {epoch if epoch is not None else 'N/A'}")
    print("1. Model performance:")
    train_acc_str = f"{train_acc}%" if train_acc is not None else "N/A"
    test_acc_str = f"{test_acc}%" if test_acc is not None else "N/A"
    print(
        f"Accuracy on entire dataset: {evaluate_model(model, dataset.dataset, dataset.labels)}%"
        f" \n\t Train set accuracy: {train_acc_str}"
        f" \n\t Test set accuracy: {test_acc_str}"
    )

    op_elbows = {}
    for op in config.operations:
        op_elbows[op] = fourier_analysis.compute_ip_elbow(
            checkpoint,
            dataset,
            op,
            norm_threshold=config.analysis.fourier_norm_threshold,
            fourier_reg_mode=config.training.fourier_reg_mode,
            analysis_cache=analysis_cache,
        )

    for op in config.operations:
        ip_elbow = op_elbows[op]
        print(f"\n=== {op.upper()} ===")

        print("2. Generating fourier coefficient analysis plots")
        fourier_analysis.fourier_spectrum_analysis_plotting(
            model,
            checkpoint,
            dataset,
            resolved_path,
            op,
            fourier_reg_mode=config.training.fourier_reg_mode,
            norm_threshold=config.analysis.fourier_norm_threshold,
            freq_threshold_top_fraction=config.analysis.freq_threshold_top_fraction,
            freq_threshold_min_components=config.analysis.freq_threshold_min_components,
            freq_threshold_fixed=config.analysis.freq_threshold_fixed,
            analysis_cache=analysis_cache,
        )
        print()

        print("3. Performing SVD ablation analysis")
        try:
            ablation_analysis.svd_ablation_analysis(
                config.model.to_dict(),
                checkpoint,
                dataset,
                op,
                ip_elbow,
                analysis_cache=analysis_cache,
            )
        except Exception as e:
            print(f"Exception: {e}")
        print()

        print("4. Performing fourier component ablation analysis")
        try:
            ablation_analysis.fourier_ablation_analysis(
                config.model.to_dict(),
                checkpoint,
                dataset,
                op,
                ip_elbow,
                analysis_cache=analysis_cache,
            )
        except Exception as e:
            print(f"Exception: {e}")
        print()

        if op in ("addition", "subtraction"):
            print("5. Trigonometric Identity Verification:")
            try:
                if op == "addition":
                    trig_analysis.verify_trigonometric_identity(
                        model,
                        checkpoint,
                        dataset,
                        freq_threshold_top_fraction=config.analysis.freq_threshold_top_fraction,
                        freq_threshold_min_components=config.analysis.freq_threshold_min_components,
                        freq_threshold_fixed=config.analysis.freq_threshold_fixed,
                        analysis_cache=analysis_cache,
                    )
                else:
                    trig_analysis.verify_trigonometric_identity_subtraction(
                        model,
                        checkpoint,
                        dataset,
                        freq_threshold_top_fraction=config.analysis.freq_threshold_top_fraction,
                        freq_threshold_min_components=config.analysis.freq_threshold_min_components,
                        freq_threshold_fixed=config.analysis.freq_threshold_fixed,
                        analysis_cache=analysis_cache,
                    )
            except Exception as e:
                print(f"Exception: {e}")
            print()

    print("\n6. Generating Weight SVD spectrum plots (whole model)")
    try:
        svd_analysis.svd_spectrum_analysis_plotting_model(
            model,
            checkpoint,
            dataset,
            resolved_path,
            op_elbows,
            config.operations,
            analysis_cache=analysis_cache,
        )
    except Exception as e:
        print(f"Exception: {e}")
    print()

    print("=== Analysis Complete ===")


In [ ]:
from pathlib import Path
from IPython.display import Image, display

# Re-bind run_dir so static analyzers can resolve it in this cell
if "run_dir" not in globals():
    raise RuntimeError("Missing run_dir. Run the setup cell first.")
run_dir = Path(globals()["run_dir"])

figures_dir = run_dir / "figures"
print("Run directory:", run_dir)
print("Has best checkpoint:", (run_dir / "best.pt").exists())
print("Has final checkpoint:", (run_dir / "final.pt").exists())

if figures_dir.exists():
    figure_files = sorted(figures_dir.glob("*.png"))
    print(f"Generated {len(figure_files)} figure(s) in {figures_dir}")

    for fig_path in figure_files:
        print(f" - {fig_path.name}")

    if figure_files:
        print("\nDisplaying figures inline:\n")
        for fig_path in figure_files:
            print(fig_path.name)
            display(Image(filename=str(fig_path)))
else:
    print("No figures directory found yet.")
